# Paper Final Benchmark (Colab H100)

This notebook is for your paper-grade benchmark run.

It does four things:
1. Mounts Google Drive and enters the repo.
2. Creates a run-specific config from `configs/paper_final_500.toml`.
3. Builds graphs (if needed).
4. Runs 500-epoch benchmarking across `ff_layerwise`, `ff_e2e`, and `backprop`, then writes a paper summary table.
Implementation checks in this notebook:
- verifies benchmark outputs include fold stability columns (mean/std/min) for economics
- validates critic-aware FF benchmark metadata columns before generating paper summary


In [1]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)


def _resolve_repo_root() -> Path:
    env_repo = os.environ.get("FRM_REPO_DIR", "").strip()
    if env_repo:
        p = Path(env_repo)
        if (p / "configs/default.toml").exists():
            return p

    candidates = [
        Path.cwd(),
        Path("/content/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/Forward-Risk-Manager"),
        Path("/content/drive/MyDrive/forward-risk-manager"),
    ]
    if IN_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            candidates.extend(
                sorted(p for p in drive_root.glob("*Forward*Risk*Manager*") if p.is_dir())
            )

    for p in candidates:
        if (p / "configs/default.toml").exists():
            return p

    raise FileNotFoundError(
        "Could not find repo root containing configs/default.toml. "
        "Set FRM_REPO_DIR or update candidate paths in this cell."
    )


ROOT = _resolve_repo_root()
os.chdir(ROOT)
print("repo root:", ROOT)
print("cwd:", Path.cwd())


Mounted at /content/drive
repo root: /content/drive/MyDrive/Forward-Risk-Manager
cwd: /content/drive/MyDrive/Forward-Risk-Manager


In [2]:
import importlib.util
import os
import re
import shlex
import subprocess
import sys
import time
from collections import deque

PYTHON_EXE = shlex.quote(sys.executable)


def run(cmd: str, allow_fail: bool = False, tail_lines: int = 200) -> bool:
    print("\n" + "=" * 120)
    print(cmd)
    print("=" * 120)
    t0 = time.time()

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

    logs_dir = ROOT / "runs" / "experiments" / "_paper_logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", cmd).strip("_")[:120] or "command"
    log_path = logs_dir / f"{int(time.time())}_{safe_name}.log"

    proc = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None

    tail = deque(maxlen=max(40, int(tail_lines)))
    with log_path.open("w", encoding="utf-8") as lf:
        for line in proc.stdout:
            print(line, end="")
            lf.write(line)
            tail.append(line.rstrip("\n"))
    rc = proc.wait()

    elapsed = time.time() - t0
    print(f"\ncompleted in {elapsed:.2f}s | log: {log_path}")
    if rc != 0:
        if tail:
            print("---- command output tail ----")
            for ln in tail:
                print(ln)
            print("---- end tail ----")
        msg = f"command failed ({rc}): {cmd}"
        if allow_fail:
            print("WARNING:", msg)
            return False
        raise RuntimeError(msg)
    return True


required_modules = [
    "torch",
    "torch_geometric",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
INSTALL_DEPS = bool(missing)

if INSTALL_DEPS:
    print("Missing modules detected:", missing)
    run(f"{PYTHON_EXE} -m pip install --upgrade pip setuptools wheel")
    run(f"{PYTHON_EXE} -m pip install -r requirements.txt")
    run(f"{PYTHON_EXE} -m pip install -e .")
else:
    print("Dependencies already available. Skipping install.")


Missing modules detected: ['torch_geometric']

/usr/bin/python3 -m pip install --upgrade pip setuptools wheel
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 104.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.

completed in 5.28s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_paper_logs/1771978821_usr_bin_python3_-m_pip_install_--upgrade_pip_setuptools_wheel.log

/usr/bin

In [3]:
from datetime import datetime, timezone
import re

TEMPLATE_CONFIG = ROOT / "configs" / "paper_final_500.toml"
assert TEMPLATE_CONFIG.exists(), f"Missing template config: {TEMPLATE_CONFIG}"

BASE_TOKEN = "runs/experiments/paper_final_500_TEMPLATE"
RUN_ID_OVERRIDE = ""  # leave empty for a fresh run
RESUME_POLICY = "auto"  # "auto" | "force_new" | "force_resume"

resume_requested = bool(RUN_ID_OVERRIDE.strip()) and RESUME_POLICY != "force_new"
if RESUME_POLICY == "force_resume" and not RUN_ID_OVERRIDE.strip():
    raise ValueError("RESUME_POLICY='force_resume' requires a non-empty RUN_ID_OVERRIDE")

if resume_requested:
    RUN_ID = RUN_ID_OVERRIDE.strip()
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID
    runtime_config = RUN_ROOT / "runtime_config.toml"
    if runtime_config.exists():
        print("resuming run id:", RUN_ID)
        print("runtime config:", runtime_config)
        print("run root:", RUN_ROOT)
    elif RESUME_POLICY == "force_resume":
        raise FileNotFoundError(f"Missing runtime config for resume: {runtime_config}")
    else:
        print(f"Resume requested but runtime config missing at {runtime_config}; creating a new run.")
        resume_requested = False

if not resume_requested:
    RUN_ID = f"paper_final_500_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

    for sub in ("data", "metrics", "plots", "logs", "models", "configs"):
        (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

    runtime_config = RUN_ROOT / "runtime_config.toml"
    cfg_txt = TEMPLATE_CONFIG.read_text()
    assert BASE_TOKEN in cfg_txt, f"Expected token {BASE_TOKEN!r} in template config"
    cfg_txt = cfg_txt.replace(BASE_TOKEN, f"runs/experiments/{RUN_ID}")
    runtime_config.write_text(cfg_txt)

    print("run id:", RUN_ID)
    print("runtime config:", runtime_config)
    print("run root:", RUN_ROOT)

APPLY_GPU_RUNTIME_OVERRIDES = True
PAPER_EXPECTED_GPU_PROFILE = "any"
PAPER_ENFORCE_EXPECTED_GPU = False

if APPLY_GPU_RUNTIME_OVERRIDES:
    try:
        import torch

        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
    except Exception:
        gpu_name = ""

    gpu_upper = gpu_name.upper()
    gpu_profile = "h100" if "H100" in gpu_upper else ("a100" if "A100" in gpu_upper else ("t4" if "T4" in gpu_upper else "generic"))
    if PAPER_EXPECTED_GPU_PROFILE not in {"", "any", "auto"} and gpu_profile != PAPER_EXPECTED_GPU_PROFILE:
        msg = (
            f"paper_final_benchmark_colab expects GPU profile '{PAPER_EXPECTED_GPU_PROFILE}' "
            f"but found '{gpu_profile}' ({gpu_name or 'unknown'})."
        )
        if PAPER_ENFORCE_EXPECTED_GPU:
            raise RuntimeError(msg)
        print("WARNING:", msg)

    amp_dtype = "bfloat16"
    auto_tune_max_batch = 128
    graph_stride = 1
    graph_limit = 0
    graph_limit_keep_recent = True
    epoch_graph_fraction = 0.5
    epoch_graph_min = 768
    epoch_graph_mode = "recent_bias"
    epoch_graph_recent_bias_alpha = 1.5
    epoch_graph_regime_change_boost = 2.0
    epoch_graph_regime_change_spread = 5
    epoch_graph_regime_loss_scale = 0.5
    epoch_graph_regime_loss_cap_percentile = 0.9
    ff_dual_neg_every_n_batches = 8
    adaptive_dual_neg_enabled = True
    adaptive_dual_neg_warmup_epochs = 5
    adaptive_dual_neg_min_every_n_batches = 4
    adaptive_dual_neg_max_every_n_batches = 16
    adaptive_dual_neg_sep_low = 0.05
    adaptive_dual_neg_sep_high = 0.12
    adaptive_dual_neg_forward_neg_share_high = 0.38
    adaptive_dual_neg_step_factor = 2.0

    cfg_txt = runtime_config.read_text()
    cfg_txt = re.sub(r'(?m)^amp_dtype\s*=\s*".*"\s*$', f'amp_dtype = "{amp_dtype}"', cfg_txt)
    cfg_txt = re.sub(r'(?m)^backprop_amp_dtype\s*=\s*".*"\s*$', f'backprop_amp_dtype = "{amp_dtype}"', cfg_txt)
    cfg_txt = re.sub(r'(?m)^auto_tune_max_batch\s*=\s*\d+\s*$', f'auto_tune_max_batch = {auto_tune_max_batch}', cfg_txt)
    train_schedule_lines = [
        f'graph_stride = {int(graph_stride)}',
        f'graph_limit = {int(graph_limit)}',
        f'graph_limit_keep_recent = {str(bool(graph_limit_keep_recent)).lower()}',
        f'epoch_graph_fraction = {float(epoch_graph_fraction)}',
        f'epoch_graph_min = {int(epoch_graph_min)}',
        f'epoch_graph_mode = "{epoch_graph_mode}"',
        f'epoch_graph_recent_bias_alpha = {float(epoch_graph_recent_bias_alpha)}',
        f'epoch_graph_regime_change_boost = {float(epoch_graph_regime_change_boost)}',
        f'epoch_graph_regime_change_spread = {int(epoch_graph_regime_change_spread)}',
        f'epoch_graph_regime_loss_scale = {float(epoch_graph_regime_loss_scale)}',
        f'epoch_graph_regime_loss_cap_percentile = {float(epoch_graph_regime_loss_cap_percentile)}',
        f'ff_dual_neg_every_n_batches = {int(ff_dual_neg_every_n_batches)}',
        f'adaptive_dual_neg_enabled = {str(bool(adaptive_dual_neg_enabled)).lower()}',
        f'adaptive_dual_neg_warmup_epochs = {int(adaptive_dual_neg_warmup_epochs)}',
        f'adaptive_dual_neg_min_every_n_batches = {int(adaptive_dual_neg_min_every_n_batches)}',
        f'adaptive_dual_neg_max_every_n_batches = {int(adaptive_dual_neg_max_every_n_batches)}',
        f'adaptive_dual_neg_sep_low = {float(adaptive_dual_neg_sep_low)}',
        f'adaptive_dual_neg_sep_high = {float(adaptive_dual_neg_sep_high)}',
        f'adaptive_dual_neg_forward_neg_share_high = {float(adaptive_dual_neg_forward_neg_share_high)}',
        f'adaptive_dual_neg_step_factor = {float(adaptive_dual_neg_step_factor)}',
    ]
    if re.search(r'(?m)^graph_stride\s*=', cfg_txt):
        cfg_txt = re.sub(r'(?m)^graph_stride\s*=\s*.*$', train_schedule_lines[0], cfg_txt)
        cfg_txt = re.sub(r'(?m)^graph_limit\s*=\s*.*$', train_schedule_lines[1], cfg_txt)
        cfg_txt = re.sub(r'(?m)^graph_limit_keep_recent\s*=\s*.*$', train_schedule_lines[2], cfg_txt)
        cfg_txt = re.sub(r'(?m)^epoch_graph_fraction\s*=\s*.*$', train_schedule_lines[3], cfg_txt)
        cfg_txt = re.sub(r'(?m)^epoch_graph_min\s*=\s*.*$', train_schedule_lines[4], cfg_txt)
        cfg_txt = re.sub(r'(?m)^epoch_graph_mode\s*=\s*.*$', train_schedule_lines[5], cfg_txt)
        cfg_txt = re.sub(r'(?m)^epoch_graph_recent_bias_alpha\s*=\s*.*$', train_schedule_lines[6], cfg_txt)
        if re.search(r'(?m)^epoch_graph_regime_change_boost\s*=', cfg_txt):
            cfg_txt = re.sub(r'(?m)^epoch_graph_regime_change_boost\s*=\s*.*$', train_schedule_lines[7], cfg_txt)
            cfg_txt = re.sub(r'(?m)^epoch_graph_regime_change_spread\s*=\s*.*$', train_schedule_lines[8], cfg_txt)
            if re.search(r'(?m)^epoch_graph_regime_loss_scale\s*=', cfg_txt):
                cfg_txt = re.sub(r'(?m)^epoch_graph_regime_loss_scale\s*=\s*.*$', train_schedule_lines[9], cfg_txt)
                if re.search(r'(?m)^epoch_graph_regime_loss_cap_percentile\s*=', cfg_txt):
                    cfg_txt = re.sub(r'(?m)^epoch_graph_regime_loss_cap_percentile\s*=\s*.*$', train_schedule_lines[10], cfg_txt)
                else:
                    cfg_txt = re.sub(
                        r'(?m)^(epoch_graph_regime_loss_scale\s*=\s*.*)$',
                        lambda m: m.group(1) + "\n" + train_schedule_lines[10],
                        cfg_txt,
                        count=1,
                    )
            else:
                cfg_txt = re.sub(
                    r'(?m)^(epoch_graph_regime_change_spread\s*=\s*.*)$',
                    lambda m: m.group(1) + "\n" + train_schedule_lines[9] + "\n" + train_schedule_lines[10],
                    cfg_txt,
                    count=1,
                )
            dual_neg_anchor = train_schedule_lines[10]
            dual_neg_extra = "\n".join(train_schedule_lines[11:])
            if re.search(r'(?m)^ff_dual_neg_every_n_batches\s*=', cfg_txt):
                cfg_txt = re.sub(r'(?m)^ff_dual_neg_every_n_batches\s*=\s*.*$', train_schedule_lines[11], cfg_txt)
                if re.search(r'(?m)^adaptive_dual_neg_enabled\s*=', cfg_txt):
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_enabled\s*=\s*.*$', train_schedule_lines[12], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_warmup_epochs\s*=\s*.*$', train_schedule_lines[13], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_min_every_n_batches\s*=\s*.*$', train_schedule_lines[14], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_max_every_n_batches\s*=\s*.*$', train_schedule_lines[15], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_sep_low\s*=\s*.*$', train_schedule_lines[16], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_sep_high\s*=\s*.*$', train_schedule_lines[17], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_forward_neg_share_high\s*=\s*.*$', train_schedule_lines[18], cfg_txt)
                    cfg_txt = re.sub(r'(?m)^adaptive_dual_neg_step_factor\s*=\s*.*$', train_schedule_lines[19], cfg_txt)
                else:
                    cfg_txt = re.sub(
                        r'(?m)^(ff_dual_neg_every_n_batches\s*=\s*.*)$',
                        lambda m: m.group(1) + "\n" + dual_neg_extra,
                        cfg_txt,
                        count=1,
                    )
            else:
                cfg_txt = re.sub(
                    rf'(?m)^({re.escape(dual_neg_anchor)}\s*)$',
                    lambda m: m.group(1) + "\n" + dual_neg_extra,
                    cfg_txt,
                    count=1,
                )
        else:
            cfg_txt = re.sub(
                r'(?m)^(epoch_graph_recent_bias_alpha\s*=\s*.*)$',
                lambda m: m.group(1) + "\n" + train_schedule_lines[7] + "\n" + train_schedule_lines[8] + "\n" + train_schedule_lines[9] + "\n" + train_schedule_lines[10] + "\n" + "\n".join(train_schedule_lines[11:]),
                cfg_txt,
                count=1,
            )
    else:
        cfg_txt = re.sub(r'(?m)^(batch_size\s*=\s*\d+\s*)$', lambda m: m.group(1) + "\n" + "\n".join(train_schedule_lines), cfg_txt, count=1)
    runtime_config.write_text(cfg_txt)

    print("gpu:", gpu_name or "<unknown>", "| profile:", gpu_profile)
    print("runtime overrides:", {
        "amp_dtype": amp_dtype,
        "auto_tune_max_batch": auto_tune_max_batch,
        "graph_stride": graph_stride,
        "epoch_graph_fraction": epoch_graph_fraction,
        "epoch_graph_min": epoch_graph_min,
        "epoch_graph_recent_bias_alpha": epoch_graph_recent_bias_alpha,
        "epoch_graph_regime_change_boost": epoch_graph_regime_change_boost,
        "epoch_graph_regime_change_spread": epoch_graph_regime_change_spread,
        "epoch_graph_regime_loss_scale": epoch_graph_regime_loss_scale,
        "epoch_graph_regime_loss_cap_percentile": epoch_graph_regime_loss_cap_percentile,
        "ff_dual_neg_every_n_batches": ff_dual_neg_every_n_batches,
        "adaptive_dual_neg_enabled": adaptive_dual_neg_enabled,
        "adaptive_dual_neg_warmup_epochs": adaptive_dual_neg_warmup_epochs,
        "adaptive_dual_neg_min_every_n_batches": adaptive_dual_neg_min_every_n_batches,
        "adaptive_dual_neg_max_every_n_batches": adaptive_dual_neg_max_every_n_batches,
        "adaptive_dual_neg_sep_low": adaptive_dual_neg_sep_low,
        "adaptive_dual_neg_sep_high": adaptive_dual_neg_sep_high,
        "adaptive_dual_neg_forward_neg_share_high": adaptive_dual_neg_forward_neg_share_high,
        "adaptive_dual_neg_step_factor": adaptive_dual_neg_step_factor,
    })


run id: paper_final_500_20260225_002036
runtime config: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/runtime_config.toml
run root: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036
gpu: NVIDIA H100 80GB HBM3 | profile: h100
runtime overrides: {'amp_dtype': 'bfloat16', 'auto_tune_max_batch': 128}


In [4]:
# Reuse the dedicated prebuilt rich graph artifact by default.
graphs_path = ROOT / "data" / "processed" / "graphs_master_ff_rich.pt"
RUN_BUILD_GRAPHS = False
FORCE_REBUILD_GRAPHS = False

if FORCE_REBUILD_GRAPHS:
    RUN_BUILD_GRAPHS = True

if RUN_BUILD_GRAPHS:
    run(f"{PYTHON_EXE} -u scripts/build_graphs.py --config {shlex.quote(str(ROOT / 'configs' / 'master_graph_ff.toml'))}")
elif graphs_path.exists():
    print(f"Using prebuilt graph artifact: {graphs_path}")
else:
    raise FileNotFoundError(
        f"Missing prebuilt graph artifact: {graphs_path}. "
        "Run notebooks/graph_factory_colab.ipynb first, or set RUN_BUILD_GRAPHS=True."
    )

cfg_txt = runtime_config.read_text()
cfg_txt = re.sub(
    r'(?m)^graphs\s*=\s*".*"\s*$',
    'graphs = "data/processed/graphs_master_ff_rich.pt"',
    cfg_txt,
    count=1,
)
runtime_config.write_text(cfg_txt)
print("Patched runtime config to use prebuilt rich graph.")



/usr/bin/python3 -u scripts/build_graphs.py --config /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/runtime_config.toml
Macro features: source=auto cols=12 rows=6546

Building graphs: 100%|██████████| 2064/2064 [01:29<00:00, 23.09win/s]
Wrote runs/experiments/paper_final_500_20260225_002036/data/graphs.pt with 2064 graphs
Date range: 2011-06-01 -> 2026-02-06 | windows: 6527 | built: 2064 | skipped_lag=0, skipped: members=4463, cols=0, min_nodes=0, no_edges=0

completed in 122.96s | log: /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/_paper_logs/1771978840_usr_bin_python3_-u_scripts_build_graphs.py_--config_content_drive_MyDrive_Forward-Risk-Manager_runs_experiments_paper_fi.log


In [5]:
import csv
import re

BENCH_MODES = ["ff_layerwise", "ff_e2e", "backprop"]
RESUME_BENCHMARK = True
FORCE_RERUN_MODES = set()  # e.g. {"backprop"}

benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
baseline_csv = RUN_ROOT / "metrics" / "benchmark_baseline.csv"

per_mode_benchmark = {
    mode: RUN_ROOT / "metrics" / f"benchmark_{mode}.csv" for mode in BENCH_MODES
}
per_mode_folds = {
    mode: RUN_ROOT / "metrics" / f"benchmark_walk_forward_folds_{mode}.csv" for mode in BENCH_MODES
}
per_mode_baseline = {
    mode: RUN_ROOT / "metrics" / f"benchmark_baseline_{mode}.csv" for mode in BENCH_MODES
}

cfg_template = runtime_config.read_text()
assert "out_csv" in cfg_template and "walk_forward_out_csv" in cfg_template, (
    "runtime config missing benchmark out paths"
)

(RUN_ROOT / "configs").mkdir(parents=True, exist_ok=True)

for mode in BENCH_MODES:
    mode_csv = per_mode_benchmark[mode]
    mode_folds = per_mode_folds[mode]
    mode_baseline = per_mode_baseline[mode]

    if (
        RESUME_BENCHMARK
        and mode not in FORCE_RERUN_MODES
        and mode_csv.exists()
        and mode_csv.stat().st_size > 0
    ):
        print(f"Skipping mode={mode}; found {mode_csv}")
        continue

    cfg_mode = cfg_template
    cfg_mode = re.sub(
        r'(?m)^out_csv\s*=\s*".*"\s*$',
        f'out_csv = "{mode_csv.as_posix()}"',
        cfg_mode,
        count=1,
    )
    cfg_mode = re.sub(
        r'(?m)^walk_forward_out_csv\s*=\s*".*"\s*$',
        f'walk_forward_out_csv = "{mode_folds.as_posix()}"',
        cfg_mode,
        count=1,
    )
    if re.search(r'(?m)^baseline_out_csv\s*=\s*".*"\s*$', cfg_mode):
        cfg_mode = re.sub(
            r'(?m)^baseline_out_csv\s*=\s*".*"\s*$',
            f'baseline_out_csv = "{mode_baseline.as_posix()}"',
            cfg_mode,
            count=1,
        )
    else:
        cfg_mode = re.sub(
            r'(?m)^out_csv\s*=\s*".*"\s*$',
            f'out_csv = "{mode_csv.as_posix()}"\nbaseline_out_csv = "{mode_baseline.as_posix()}"',
            cfg_mode,
            count=1,
        )

    cfg_mode_path = RUN_ROOT / "configs" / f"runtime_config_{mode}.toml"
    cfg_mode_path.write_text(cfg_mode)

    run(
        f"{PYTHON_EXE} -u scripts/benchmark_training.py "
        f"--config {shlex.quote(str(cfg_mode_path))} "
        f"--modes {mode}"
    )


def combine_csv(paths, out_path):
    rows = []
    fieldnames = []
    for p in paths:
        if not p.exists() or p.stat().st_size == 0:
            continue
        with p.open("r", newline="") as f:
            reader = csv.DictReader(f)
            if reader.fieldnames:
                for k in reader.fieldnames:
                    if k not in fieldnames:
                        fieldnames.append(k)
            for row in reader:
                rows.append(row)

    if not rows:
        raise RuntimeError(f"No rows found to combine into {out_path}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    print(f"Wrote combined CSV: {out_path} rows={len(rows)}")


combine_csv([per_mode_benchmark[m] for m in BENCH_MODES], benchmark_csv)
combine_csv([per_mode_folds[m] for m in BENCH_MODES], folds_csv)
if any(p.exists() and p.stat().st_size > 0 for p in per_mode_baseline.values()):
    combine_csv([per_mode_baseline[m] for m in BENCH_MODES], baseline_csv)
else:
    print("No per-mode baseline CSVs found; skipping combined baseline file.")



/usr/bin/python3 -u scripts/benchmark_training.py --config /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/configs/runtime_config_ff_layerwise.toml --modes ff_layerwise
econ ticker: requested=SPY,MDY,BWC,AAON,AUTO effective=BWC source=requested_priority rows=10439
walk-forward splits=3 (train_frac=0.6, eval_frac=0.2, step_frac=0.1)
benchmark resiliency: retry_safe_on_error=True, continue_on_mode_error=True, torch_compile=False (mode=max-autotune-no-cudagraphs)

Benchmark: 100%|██████████| 500/500 [09:40<00:00,  1.16s/epoch]
calibrated goodness_target=8.8018 (train-cal acc=0.8425)

Benchmark: 100%|██████████| 500/500 [11:18<00:00,  1.36s/epoch]
calibrated goodness_target=14.8055 (train-cal acc=0.8449)

Benchmark: 100%|██████████| 500/500 [12:46<00:00,  1.53s/epoch]
calibrated goodness_target=63.9475 (train-cal acc=0.4812)
Wrote /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark_ff_l

In [6]:
benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
baseline_csv = RUN_ROOT / "metrics" / "benchmark_baseline.csv"
summary_md = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
summary_csv = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
summary_json = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

import pandas as pd

assert benchmark_csv.exists(), f"Missing benchmark CSV: {benchmark_csv}"
bench_df = pd.read_csv(benchmark_csv)
required_cols = [
    "mode",
    "row_type",
    "objective_track",
    "econ_sharpe_uplift",
    "econ_sharpe_uplift_std",
    "econ_sharpe_uplift_min",
    "risk_head_enabled_effective",
    "portfolio_head_enabled_effective",
    "econ_regime_gate_enabled",
    "econ_regime_confidence_mean",
    "econ_regime_exposure_mean",
]
missing = [c for c in required_cols if c not in bench_df.columns]
assert not missing, f"Benchmark CSV missing expected columns: {missing}"
print("benchmark column check passed:", required_cols)

recommended_cols = [
    "econ_oos_sharpe_uplift_min",
    "econ_oos_folds_used",
    "econ_signal_polarity",
    "econ_regime_thresholding_enabled",
]
missing_recommended = [c for c in recommended_cols if c not in bench_df.columns]
if missing_recommended:
    print("WARNING: benchmark is missing newer econ diagnostics columns:", missing_recommended)

run(
    f"{PYTHON_EXE} -u scripts/paper_benchmark_summary.py "
    f"--benchmark {shlex.quote(str(benchmark_csv))} "
    f"--folds {shlex.quote(str(folds_csv))} "
    f"--out-md {shlex.quote(str(summary_md))} "
    f"--out-csv {shlex.quote(str(summary_csv))} "
    f"--out-json {shlex.quote(str(summary_json))}"
)

print("benchmark:", benchmark_csv)
print("folds:", folds_csv)
print("baseline:", baseline_csv, "exists=", baseline_csv.exists())
print("summary md:", summary_md)
print("summary csv:", summary_csv)
print("summary json:", summary_json)


benchmark column check passed: ['mode', 'row_type', 'objective_track', 'econ_sharpe_uplift', 'econ_sharpe_uplift_std', 'econ_sharpe_uplift_min', 'risk_head_enabled_effective', 'portfolio_head_enabled_effective', 'econ_regime_gate_enabled', 'econ_regime_confidence_mean', 'econ_regime_exposure_mean']

/usr/bin/python3 -u scripts/paper_benchmark_summary.py --benchmark /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark.csv --folds /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark_walk_forward_folds.csv --out-md /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/logs/paper_benchmark_summary.md --out-csv /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/paper_benchmark_summary.csv --out-json /content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_00203

In [7]:
from IPython.display import Markdown, display

if summary_md.exists():
    display(Markdown(summary_md.read_text()))
else:
    print("Missing summary markdown:", summary_md)

print("\nArtifact check:")
for path in [benchmark_csv, folds_csv, baseline_csv, summary_md, summary_csv, summary_json]:
    print(f"{path} -> exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

# Paper Benchmark Summary

Source: `/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark.csv`

| mode | quality_metric | quality_value | eval_auroc | eval_auprc | avg_epoch_s | graphs_per_s | econ_sharpe_uplift | econ_ann_return_uplift |
|---|---|---|---|---|---|---|---|---|
| ff_layerwise | eval_sep | 6.5120 | 0.6283 | 0.6707 | 1.349 | 1070.2 | -0.223 | -26.70% |
| ff_e2e | eval_sep | 14.2377 | 1.0000 | 1.0000 | 0.706 | 2045.5 | -0.338 | -27.76% |
| backprop | eval_auroc | 1.0000 | 1.0000 | 1.0000 | 0.340 | 4253.3 | -0.448 | -29.28% |

## Key Points
- Fastest: `backprop` (4253.3 graphs/s).
- Best quality: `ff_e2e` (eval_sep=14.2377).
- Best economics: `ff_layerwise` (Sharpe uplift=-0.223, ann return uplift=-26.70%).



Artifact check:
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark.csv -> exists=True size=27147
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark_walk_forward_folds.csv -> exists=True size=16681
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/benchmark_baseline.csv -> exists=True size=546
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/logs/paper_benchmark_summary.md -> exists=True size=785
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/metrics/paper_benchmark_summary.csv -> exists=True size=1190
/content/drive/MyDrive/Forward-Risk-Manager/runs/experiments/paper_final_500_20260225_002036/logs/paper_benchmark_summary.json -> exists=True size=7055
